In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

In [20]:
train_df = pd.read_csv("train_runes.csv")
train_df.head()

,rune,spell
0,WWWWF,1
1,FWEFW,1
2,FFEFF,0
3,EWFWE,1
4,EEEWF,0


In [21]:
test_df = pd.read_csv("test_runes.csv")
test_df.head()

,rune
0,FWFEF
1,EFEEE
2,EFWFE
3,FEWWF
4,WFEFE


In [22]:
# =========================
# 2. Feature engineering
# =========================
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()
    
    # Разбиваем строку rune на позиции
    for i in range(5):
        X[f"r{i+1}"] = X["rune"].str[i]
    
    # Биграммы
    for i in range(4):
        X[f"bg_{i+1}_{i+2}"] = X["rune"].str[i:i+2]
    
    # Триграммы
    for i in range(3):
        X[f"tg_{i+1}_{i+3}"] = X["rune"].str[i:i+3]
    
    # Подсчет символов
    X["count_F"] = X["rune"].str.count("F")
    X["count_W"] = X["rune"].str.count("W")
    X["count_E"] = X["rune"].str.count("E")
    
    # Equality features
    X["eq_1_2"] = (X["r1"] == X["r2"]).astype(int)
    X["eq_2_3"] = (X["r2"] == X["r3"]).astype(int)
    X["eq_3_4"] = (X["r3"] == X["r4"]).astype(int)
    X["eq_4_5"] = (X["r4"] == X["r5"]).astype(int)
    
    X["eq_1_3"] = (X["r1"] == X["r3"]).astype(int)
    X["eq_1_4"] = (X["r1"] == X["r4"]).astype(int)
    X["eq_1_5"] = (X["r1"] == X["r5"]).astype(int)
    X["eq_2_4"] = (X["r2"] == X["r4"]).astype(int)
    X["eq_2_5"] = (X["r2"] == X["r5"]).astype(int)
    X["eq_3_5"] = (X["r3"] == X["r5"]).astype(int)
    
    # Доп. полезные фичи
    X["is_palindrome"] = (
        (X["r1"] == X["r5"]) & (X["r2"] == X["r4"])
    ).astype(int)
    
    X["n_transitions"] = (
        (X["r1"] != X["r2"]).astype(int)
        + (X["r2"] != X["r3"]).astype(int)
        + (X["r3"] != X["r4"]).astype(int)
        + (X["r4"] != X["r5"]).astype(int)
    )
    
    return X


train_feat = build_features(train_df)
test_feat = build_features(test_df)
train_feat.head()

,rune,spell,r1,r2,r3,r4,r5,bg_1_2,bg_2_3,bg_3_4,...,eq_3_4,eq_4_5,eq_1_3,eq_1_4,eq_1_5,eq_2_4,eq_2_5,eq_3_5,is_palindrome,n_transitions
0,WWWWF,1,W,W,W,W,F,WW,WW,WW,...,1,0,1,1,0,1,0,0,0,1
1,FWEFW,1,F,W,E,F,W,FW,WE,EF,...,0,0,0,1,0,0,1,0,0,4
2,FFEFF,0,F,F,E,F,F,FF,FE,EF,...,0,1,0,1,1,1,1,0,1,2
3,EWFWE,1,E,W,F,W,E,EW,WF,FW,...,0,0,0,0,1,1,0,0,1,4
4,EEEWF,0,E,E,E,W,F,EE,EE,EW,...,0,0,1,0,0,0,0,0,0,2


In [23]:
y = train_feat["spell"]
X = train_feat.drop(columns=["spell"])
X_test = test_feat.copy()

In [24]:
# rune оставим тоже как категориальную фичу
categorical_features = [
    "rune",
    "r1", "r2", "r3", "r4", "r5",
    "bg_1_2", "bg_2_3", "bg_3_4", "bg_4_5",
    "tg_1_3", "tg_2_4", "tg_3_5",
]

numeric_features = [
    "count_F", "count_W", "count_E",
    "eq_1_2", "eq_2_3", "eq_3_4", "eq_4_5",
    "eq_1_3", "eq_1_4", "eq_1_5",
    "eq_2_4", "eq_2_5", "eq_3_5",
    "is_palindrome", "n_transitions",
]

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

Categorical: ['rune', 'r1', 'r2', 'r3', 'r4', 'r5', 'bg_1_2', 'bg_2_3', 'bg_3_4', 'bg_4_5', 'tg_1_3', 'tg_2_4', 'tg_3_5']
Numeric: ['count_F', 'count_W', 'count_E', 'eq_1_2', 'eq_2_3', 'eq_3_4', 'eq_4_5', 'eq_1_3', 'eq_1_4', 'eq_1_5', 'eq_2_4', 'eq_2_5', 'eq_3_5', 'is_palindrome', 'n_transitions']


In [25]:
# =========================
# 3. Препроцессинг
# =========================
preprocessor_for_linear = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
    ]
)

preprocessor_for_trees = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
    ]
)


# =========================
# 4. Модели
# =========================
models = {
    "logreg": Pipeline([
        ("preprocessor", preprocessor_for_linear),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    
    "random_forest": Pipeline([
        ("preprocessor", preprocessor_for_trees),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            random_state=42,
        )),
    ]),
    
    "extra_trees": Pipeline([
        ("preprocessor", preprocessor_for_trees),
        ("model", ExtraTreesClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=1,
            random_state=42,
        )),
    ]),
}


# =========================
# 5. CV
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy", n_jobs=-1)
    scores[name] = cv_scores.mean()
    print(f"{name}: mean={cv_scores.mean():.4f}, std={cv_scores.std():.4f}, all={cv_scores}")

best_model_name = max(scores, key=scores.get)
best_model = models[best_model_name]

print("\nBest model:", best_model_name, "CV accuracy:", scores[best_model_name])

logreg: mean=0.9706, std=0.0186, all=[0.97058824 0.94117647 0.97058824 0.97058824 1.        ]
random_forest: mean=0.9824, std=0.0235, all=[1.         0.94117647 0.97058824 1.         1.        ]
extra_trees: mean=0.9706, std=0.0263, all=[1.         0.94117647 0.94117647 0.97058824 1.        ]

Best model: random_forest CV accuracy: 0.9823529411764707


In [27]:
# =========================
# 6. Обучение на всем train
# =========================
best_model.fit(X, y)

test_pred = best_model.predict(X_test)

# Если в test есть id — сохраним его
if "id" in test_df.columns:
    submission = pd.DataFrame({
        "id": test_df["id"],
        "spell": test_pred
    })
else:
    submission = pd.DataFrame({
        "spell": test_pred
    })

submission.to_csv("answers.csv", index=False)
print("\nanswers.csv saved")
print(submission.head())


answers.csv saved
   spell
0      1
1      0
2      0
3      0
4      0


In [35]:
n, k = map(int, input().split())
platforms = list(map(int, input().split()))
print(n,k, platforms)

5 3 [2, 3, 4]


In [36]:
platforms = list(map(int, input().split())) + [n]
print(platforms)

[4, 5, 6, 5]


In [37]:
def optimal_path():
    n, k = map(int, input().split())
    platforms = list(map(int, input().split())) + [n]

    dp = dict()
    dp[0] = ''

    for i, platform in enumerate(platforms):

        if (platform - 1) in dp and (platform - 2) in dp:
            if len(dp[platform - 1]) > len(dp[platform - 2]):
                dp[platform] = dp[platform - 2] + '2'
            else:
                dp[platform] = dp[platform - 1] + '1'
        elif (platform - 1) in dp:
            dp[platform] = dp[platform - 1] + '1'
        elif (platform - 2) in dp:
            dp[platform] = dp[platform - 2] + '2'
        else:
            print(-1)
            return


    print(len(dp[n]))
    print(dp[n])
optimal_path()

3
221


In [39]:
for i, platform in enumerate([1,23,4,5,23,53,54,67][1:], 1):
    print(i, platform)

1 23
2 4
3 5
4 23
5 53
6 54
7 67
